Exercise 1 — Customer Review to Support Ticket Pipeline
Build a 2-step pipeline. Step 1 extracts the core complaint, product/feature mentioned, and customer sentiment from a raw customer review as structured data. Step 2 generates a support ticket summary for the internal team, using only the structured data from Step 1, not the raw review text.

Input: review_text

Output: the structured complaint data from Step 1, and the support ticket summary from Step 2

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: complaint, product_or_feature, sentiment)

Step 2 — Template: ChatPromptTemplate (system persona: "support ticket writer")  |  Parser: StrOutputParser

In [2]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

What each import does:
- BaseModel: The base class for defining our output schema.
- Field: Describes the purpose of each field.
- PromptTemplate: Creates a prompt containing variables.
- PydanticOutputParser: Converts the LLM's response into our Pydantic model.

In [3]:
class ComplaintData(BaseModel):
    complaint: str = Field(
        description="The main problem reported by the customer"
    )

    product_or_feature: str = Field(
        description="The product, page, or feature involved"
    )

    sentiment: str = Field(
        description="The emotional tone expressed by the customer"
    )

What happens here?

We're defining the structure of the data we expect, not extracting the data itself.

For example, complaint: str means the complaint must be represented as a string. The Field descriptions tell us what each field means and help guide the model when the parser's format instructions are included in the prompt.

Notice that sentiment is also a string. We haven't restricted it to a fixed list such as Positive, Neutral, and Negative; the model can use a more descriptive label such as Frustrated.

In [4]:
complaint_parser = PydanticOutputParser(
    pydantic_object=ComplaintData
)

The parser needs to know which schema it should use. Here, we've supplied ComplaintData.

In [5]:
print(complaint_parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"complaint": {"description": "The main problem reported by the customer", "title": "Complaint", "type": "string"}, "product_or_feature": {"description": "The product, page, or feature involved", "title": "Product Or Feature", "type": "string"}, "sentiment": {"description": "The emotional tone expressed by the customer", "title": "Sentiment", "type": "string"}}, "required": ["complaint", "product_or_feature", "sentiment"]}
```


In [6]:
complaint_prompt = PromptTemplate(
    template="""
You are extracting information from a customer review.

Extract these three fields:
1. complaint
2. product_or_feature
3. sentiment

Use only information supported by the review.
Do not invent missing details.

Customer review:
{review_text}

{format_instructions}
""",
    input_variables=["review_text"],
    partial_variables={
        "format_instructions":
            complaint_parser.get_format_instructions()
    }
)

In [13]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

In [14]:
step1_chain = complaint_prompt | llm | complaint_parser

In [15]:
review_text = """
I've been trying to download my invoice from the billing
page for three days. Every time I click the download button,
the page crashes. This is really frustrating!
"""

complaint_data = step1_chain.invoke({
    "review_text": review_text
})

print(complaint_data)

complaint='The download button on the billing page crashes every time I try to download my invoice.' product_or_feature='billing page' sentiment='frustrated'


## To access individual fields:

In [16]:
print("Complaint:", complaint_data.complaint)
print("Product/Feature:", complaint_data.product_or_feature)
print("Sentiment:", complaint_data.sentiment)

Complaint: The download button on the billing page crashes every time I try to download my invoice.
Product/Feature: billing page
Sentiment: frustrated


In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [18]:
ticket_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a support ticket writer.

        Write a concise internal support ticket using only
        the structured complaint data provided.

        Include:
        - Main complaint
        - Affected product or feature
        - Customer sentiment

        Do not invent an account ID, error code, root cause,
        or any other unsupported detail.
        """
    ),
    (
        "user",
        """
        Structured complaint data:
        {complaint_data}

        Write the support ticket summary.
        """
    )
])

In [19]:
ticket_parser = StrOutputParser()

step2_chain = ticket_prompt | llm | ticket_parser

In [20]:
ticket_summary = step2_chain.invoke({
    "complaint_data": complaint_data.model_dump_json(indent=2)
})

print(ticket_summary)

**Support Ticket Summary**

- **Main Complaint:** The download button on the billing page crashes every time I try to download my invoice.  
- **Affected Product/Feature:** Billing page  
- **Customer Sentiment:** Frustrated


In [21]:
def process_customer_review(review_text: str):
    # Step 1: Extract structured complaint data
    complaint_data = step1_chain.invoke({
        "review_text": review_text
    })

    # Step 2: Generate ticket using only Step 1's output
    ticket_summary = step2_chain.invoke({
        "complaint_data": complaint_data.model_dump_json(
            indent=2
        )
    })

    return {
        "complaint_data": complaint_data,
        "ticket_summary": ticket_summary
    }

In [22]:
result = process_customer_review(review_text)

print("STEP 1: STRUCTURED COMPLAINT")
print(result["complaint_data"].model_dump_json(indent=2))

print("\nSTEP 2: SUPPORT TICKET")
print(result["ticket_summary"])

STEP 1: STRUCTURED COMPLAINT
{
  "complaint": "The download button on the billing page crashes every time I try to download my invoice.",
  "product_or_feature": "billing page",
  "sentiment": "frustrated"
}

STEP 2: SUPPORT TICKET
**Support Ticket Summary**

- **Main Complaint:** The download button on the billing page crashes every time I try to download my invoice.  
- **Affected Product/Feature:** Billing page  
- **Customer Sentiment:** Frustrated
